# WoundTrack: Main Pipeline
**AI-Powered Wound Healing Assessment using MedGemma**

This notebook implements the complete WoundTrack pipeline:
1. Data loading and preprocessing
2. MedGemma-based wound analysis
3. Longitudinal progression tracking
4. Report generation and visualizations

## Setup & Installation

In [ ]:
# Install required packages
!pip install -q transformers accelerate torch pillow matplotlib plotly reportlab pandas

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from datetime import datetime, timedelta

# Check GPU availability
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Data Loading

**Kaggle Datasets to Add:**
- `leoscode/wound-segmentation-images` (2,760 samples)
- `laithjj/diabetic-foot-ulcer-dfu`

Click **"+ Add Data"** in the right panel and search for these datasets.

In [ ]:
# Dataset paths (Kaggle mounts datasets in /kaggle/input/)
WOUND_SEG_PATH = Path('/kaggle/input/wound-segmentation-images')
DFU_PATH = Path('/kaggle/input/diabetic-foot-ulcer-dfu')

# Check if datasets are available
print("Available datasets:")
if WOUND_SEG_PATH.exists():
    print(f"✓ Wound Segmentation: {len(list(WOUND_SEG_PATH.rglob('*.jpg')) + list(WOUND_SEG_PATH.rglob('*.png')))} images")
else:
    print("✗ Wound Segmentation dataset not found. Please add it via '+ Add Data'")

if DFU_PATH.exists():
    print(f"✓ DFU Dataset: {len(list(DFU_PATH.rglob('*.jpg')) + list(DFU_PATH.rglob('*.png')))} images")
else:
    print("✗ DFU dataset not found. Please add it via '+ Add Data'")

## 2. Image Preprocessing Pipeline

In [ ]:
class WoundImagePreprocessor:
    """Standardize wound images for analysis"""
    
    def __init__(self, target_size=(512, 512)):
        self.target_size = target_size
    
    def preprocess(self, image_path):
        """Load and preprocess a single image"""
        img = Image.open(image_path).convert('RGB')
        
        # Resize while maintaining aspect ratio
        img.thumbnail(self.target_size, Image.Resampling.LANCZOS)
        
        # Create padded square image
        padded = Image.new('RGB', self.target_size, (0, 0, 0))
        offset = ((self.target_size[0] - img.size[0]) // 2,
                  (self.target_size[1] - img.size[1]) // 2)
        padded.paste(img, offset)
        
        return padded
    
    def batch_preprocess(self, image_paths, output_dir):
        """Preprocess multiple images"""
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        
        processed = []
        for img_path in image_paths:
            processed_img = self.preprocess(img_path)
            output_path = output_dir / img_path.name
            processed_img.save(output_path)
            processed.append(output_path)
        
        return processed

# Initialize preprocessor
preprocessor = WoundImagePreprocessor()
print("✓ Preprocessor initialized")

## 3. Select Baseline Wound Images

Select 30-50 diverse wounds as "Day 0" baselines for synthetic progression

In [ ]:
# Collect all available wound images
all_images = []
if WOUND_SEG_PATH.exists():
    all_images.extend(list(WOUND_SEG_PATH.rglob('*.jpg')))
    all_images.extend(list(WOUND_SEG_PATH.rglob('*.png')))

if DFU_PATH.exists():
    all_images.extend(list(DFU_PATH.rglob('*.jpg')))
    all_images.extend(list(DFU_PATH.rglob('*.png')))

print(f"Total images found: {len(all_images)}")

# Select 50 random images as baselines (reproducible)
np.random.seed(42)
baseline_images = np.random.choice(all_images, min(50, len(all_images)), replace=False)

print(f"Selected {len(baseline_images)} baseline images")

# Display sample images
fig, axes = plt.subplots(2, 4, figsize=(15, 8))
for idx, ax in enumerate(axes.flat):
    if idx < len(baseline_images):
        img = Image.open(baseline_images[idx])
        ax.imshow(img)
        ax.set_title(f"Baseline {idx+1}", fontsize=10)
        ax.axis('off')
plt.tight_layout()
plt.show()

## 4. MedGemma Setup

Load MedGemma models for wound analysis

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Note: MedGemma requires authentication with Hugging Face
# You may need to set HF_TOKEN in Kaggle Secrets

MODEL_4B = "google/medgemma-1.5-4b"  # For detailed wound description
MODEL_27B = "google/medgemma-1.5-27b"  # For classification

print("Loading MedGemma 4B for wound description...")
# tokenizer_4b = AutoTokenizer.from_pretrained(MODEL_4B)
# model_4b = AutoModelForCausalLM.from_pretrained(MODEL_4B, device_map="auto")

print("Note: Uncomment above lines and add your HF token to use MedGemma")
print("For now, we'll use placeholder analysis")

## 5. Wound Description Extraction (MedGemma 4B)

Analyze wound images to extract structured information

In [ ]:
WOUND_DESCRIPTION_PROMPT = """
Analyze this wound image and provide structured details:

1. Wound Dimensions: Estimate length × width in cm
2. Tissue Types: Percentages of granulation, slough, eschar, epithelial
3. Exudate: Amount (none/minimal/moderate/heavy) and type (serous/sanguineous/purulent)
4. Surrounding Skin: Condition (intact/macerated/erythematous/indurated)
5. Wound Bed: Color and appearance

Provide measurements and observations in JSON format:
{
  "dimensions": {"length_cm": X, "width_cm": Y},
  "tissue_composition": {"granulation": %, "slough": %, "eschar": %},
  "exudate": {"amount": "...", "type": "..."},
  "surrounding_skin": "...",
  "wound_bed_color": "..."
}
"""

def analyze_wound_image(image_path, use_mock=True):
    """Extract wound characteristics using MedGemma 4B"""
    
    if use_mock:
        # Mock analysis for development
        return {
            "dimensions": {"length_cm": np.random.uniform(2, 8), "width_cm": np.random.uniform(1.5, 6)},
            "tissue_composition": {
                "granulation": np.random.randint(40, 80),
                "slough": np.random.randint(10, 40),
                "eschar": np.random.randint(0, 20)
            },
            "exudate": {
                "amount": np.random.choice(["minimal", "moderate", "heavy"]),
                "type": "serous"
            },
            "surrounding_skin": "intact",
            "wound_bed_color": "pink-red"
        }
    
    # TODO: Implement actual MedGemma inference
    # inputs = tokenizer_4b(WOUND_DESCRIPTION_PROMPT, return_tensors="pt").to(device)
    # outputs = model_4b.generate(**inputs, max_length=512)
    # return parse_json_response(outputs)

# Test on first baseline
sample_analysis = analyze_wound_image(baseline_images[0])
print("Sample wound analysis:")
print(json.dumps(sample_analysis, indent=2))

## 6. Create Synthetic Temporal Sequences

**Note**: Use the separate `synthetic_wound_generator.ipynb` (Colab) to generate healing progressions.
Upload generated sequences here for analysis.

In [ ]:
# Placeholder for synthetic sequence organization
# After generating synthetic images, load them here

def create_wound_sequence_metadata(baseline_img, num_timepoints=4):
    """Create metadata for a wound sequence"""
    wound_id = f"wound_{baseline_img.stem}"
    
    sequence = []
    for i in range(num_timepoints):
        sequence.append({
            "wound_id": wound_id,
            "timepoint": i,
            "day": i * 7,  # Weekly intervals
            "image_path": str(baseline_img) if i == 0 else f"synthetic/{wound_id}_day{i*7}.png",
            "timestamp": (datetime.now() + timedelta(days=i*7)).isoformat()
        })
    
    return sequence

# Example sequence
example_seq = create_wound_sequence_metadata(Path(baseline_images[0]))
print("Example wound sequence metadata:")
print(json.dumps(example_seq, indent=2))

## 7. Longitudinal Analysis

Compare timepoints to track healing progression

In [ ]:
def calculate_area_change(analysis_t0, analysis_t1):
    """Calculate percentage change in wound area"""
    area_t0 = analysis_t0['dimensions']['length_cm'] * analysis_t0['dimensions']['width_cm']
    area_t1 = analysis_t1['dimensions']['length_cm'] * analysis_t1['dimensions']['width_cm']
    
    change_percent = ((area_t1 - area_t0) / area_t0) * 100
    return change_percent

def calculate_tissue_shift(analysis_t0, analysis_t1):
    """Calculate changes in tissue composition"""
    shifts = {}
    for tissue_type in ['granulation', 'slough', 'eschar']:
        change = analysis_t1['tissue_composition'].get(tissue_type, 0) - \
                 analysis_t0['tissue_composition'].get(tissue_type, 0)
        shifts[tissue_type] = change
    return shifts

# Mock longitudinal comparison
analysis_day0 = analyze_wound_image(baseline_images[0])
analysis_day7 = analyze_wound_image(baseline_images[0])  # Mock

area_change = calculate_area_change(analysis_day0, analysis_day7)
tissue_shifts = calculate_tissue_shift(analysis_day0, analysis_day7)

print(f"Area change: {area_change:.1f}%")
print(f"Tissue composition shifts: {tissue_shifts}")

## 8. Classification (MedGemma 27B)

Classify wound status: Improving / Stable / Worsening

In [ ]:
CLASSIFICATION_PROMPT_TEMPLATE = """
Given the wound progression data and nurse notes, classify the healing status:

Timepoint 1 (Day {day0}): {desc0}
Timepoint 2 (Day {day1}): {desc1}
Nurse Notes: {notes}

Classification: Improving / Stable / Worsening
Confidence: 0-100%
Rationale: [brief explanation]

Guidelines:
- Improving: Wound area decreasing, granulation increasing, exudate decreasing
- Stable: Minimal changes in dimensions and tissue composition
- Worsening: Wound expanding, necrotic tissue increasing, signs of infection

Respond in JSON format:
{
  "classification": "...",
  "confidence": X,
  "rationale": "..."
}
"""

def classify_wound_status(analysis_t0, analysis_t1, nurse_notes="", use_mock=True):
    """Classify wound healing status using MedGemma 27B"""
    
    if use_mock:
        area_change = calculate_area_change(analysis_t0, analysis_t1)
        
        if area_change < -10:
            classification = "Improving"
            confidence = 85
        elif area_change > 10:
            classification = "Worsening"
            confidence = 80
        else:
            classification = "Stable"
            confidence = 75
        
        return {
            "classification": classification,
            "confidence": confidence,
            "rationale": f"Wound area changed by {area_change:.1f}%"
        }
    
    # TODO: Implement MedGemma 27B inference

# Test classification
classification = classify_wound_status(analysis_day0, analysis_day7)
print("Wound classification:")
print(json.dumps(classification, indent=2))

## 9. Visualization

Create timeline visualizations of wound progression

In [ ]:
def plot_wound_progression(sequence_data):
    """Plot wound area and tissue composition over time"""
    
    days = [d['day'] for d in sequence_data]
    areas = [d['analysis']['dimensions']['length_cm'] * d['analysis']['dimensions']['width_cm'] 
             for d in sequence_data]
    
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=days, y=areas,
        mode='lines+markers',
        name='Wound Area (cm²)',
        line=dict(color='red', width=3)
    ))
    
    fig.update_layout(
        title='Wound Area Progression',
        xaxis_title='Days',
        yaxis_title='Area (cm²)',
        template='plotly_white',
        height=400
    )
    
    return fig

# Mock sequence data
mock_sequence = [
    {"day": 0, "analysis": {"dimensions": {"length_cm": 5, "width_cm": 3}}},
    {"day": 7, "analysis": {"dimensions": {"length_cm": 4.5, "width_cm": 2.8}}},
    {"day": 14, "analysis": {"dimensions": {"length_cm": 3.8, "width_cm": 2.3}}},
    {"day": 21, "analysis": {"dimensions": {"length_cm": 3.0, "width_cm": 1.8}}}
]

fig = plot_wound_progression(mock_sequence)
fig.show()

## 10. Next Steps

✅ **Completed**:
- Dataset loading
- Preprocessing pipeline
- Mock analysis framework

📋 **To Do**:
1. Generate synthetic wound progressions (use separate Colab notebook)
2. Implement actual MedGemma inference (requires HF authentication)
3. Create PDF report generator
4. Build alert system
5. Evaluation metrics